# Perfusion Tutorial
The cell culture simulator runs batch, fed-batch **and** continuous perfusion with one model. In perfusion, fresh medium is fed continuously, cell-free harvest leaves through a cell-retention device (ATF/TFF) and a bleed removes culture to control the cell density; the volume stays constant.

Every species follows the reactor balance of Richelle et al. (2022):

$$\frac{dc}{dt} = r(c) + D_{feed}\,c_{medium} - D_{bleed}\,c - D_{harvest}\,\sigma\,c$$

with dilution rates $D = F/V$ and the passage coefficient $\sigma$ of the retention device (cells about 0, metabolites 1, product 0.8-1). The cell model includes Richelle's lysed cells and the inhibitory **biomaterial** by-product, which accumulates in fed-batch and is washed out in perfusion; that is what lets perfusion reach much higher cell densities.

A **controller** sets the feed and bleed rates every control interval from what the analytics measure. Built in: constant rates, schedules, a cell-specific perfusion rate (CSPR) and the paper's PI bleed controller for a viable-cell-density setpoint. For your own control (PI, MPC, reinforcement learning), step the reactor yourself with `simulator.start(...)` (section 3).

*A. Richelle et al., "Model-based intensification of CHO cell cultures: One-step strategy from fed-batch to perfusion," Front. Bioeng. Biotechnol. 10:948905, 2022.*

In [ ]:
from dataclasses import replace

import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from dbiolab.mammalian_cellculture_simulation import (
    CellType,
    ControlAction,
    FeedConcentrations,
    MammalianCellcultureReactor,
    MammalianRecipe,
    NoiseParameters,
    PerfusionSettings,
    RetentionDevice,
)
from dbiolab.mammalian_cellculture_simulation.control import (
    ConstantRates,
    CsprController,
    PIBleedController,
)
from dbiolab.mammalian_cellculture_simulation.presets import bench_perfusion_reactor
from dbiolab.mammalian_cellculture_simulation.scale_library import BASE_CELLTYPE

MOLAR_MASS = 150.0  # g/mmol antibody: 1 mM = 150 g/L

## 1. Setup
A 2 L bench perfusion reactor, the base CHO cell line, a perfusion medium richer in glucose and glutamine than a batch medium, and an ATF-like retention device that holds back 99 % of the cells and lets the product pass.

In [ ]:
reactor = bench_perfusion_reactor(2.0)
simulator = MammalianCellcultureReactor(reactor)
celltype = CellType.from_dict(BASE_CELLTYPE)


def concentrations(**values) -> FeedConcentrations:
    amounts = {key: 0.0 for key in FeedConcentrations.__dataclass_fields__}
    amounts.update(values)
    return FeedConcentrations(**amounts)


medium = concentrations(c_glc=40.0, c_gln=6.0, c_o2=0.2, c_co2=1.228)
retention = RetentionDevice(cell_retention=0.99, product_passage=0.95, hcp_passage=1.0)


def recipe(controller=None, name="run", runtime=480.0) -> MammalianRecipe:
    return MammalianRecipe(
        name=name,
        celltype=celltype,
        temperature=36.5,
        ph_value=7.0,
        stirring_rate=350.0,
        initial_conditions=concentrations(
            x_total=0.8e6, x_viable=0.8e6, c_glc=30.0, c_gln=4.0, c_o2=0.2, c_co2=1.228
        ),
        feeding_strategy=[],
        runtime=runtime,
        measurement_intervals=4.0,
        noise_parameters=NoiseParameters(),
        do_setpoint=0.4,
        perfusion=None
        if controller is None
        else PerfusionSettings(
            medium=medium, retention=retention, controller=controller, control_interval=2.0
        ),
    )

## 2. Batch vs. Perfusion Strategies
- **Batch**: no flows; growth stops when glucose runs out and by-products accumulate.
- **Perfusion 1.5 vvd**: constant perfusion, no bleed: the cells grow until oxygen and CO2 transfer limit them.
- **CSPR 40 pL/cell/day**: the perfusion rate follows the cell density (max. 4 vvd).
- **PI bleed, 50e6 cells/mL**: constant 2.5 vvd; the bleed holds the viable cell density at the setpoint (Richelle et al., Kp = -1 vvd per 1e6 cells/mL, Ti = 12 h), a steady state for continuous manufacturing.

In [ ]:
strategies = {
    "Batch": None,
    "Perfusion 1.5 vvd": ConstantRates(feed_vvd=1.5, bleed_vvd=0.0),
    "CSPR 40 pL/cell/d": CsprController(cspr=40.0, min_vvd=0.3, max_vvd=4.0, bleed_vvd=0.0),
    "PI bleed, 50e6/mL": PIBleedController(
        vcd_setpoint=50e6, feed_vvd=2.5, kp=-1.0, ti=12.0, cspr=0.0, filter_time=0.0
    ),
}
results = {name: simulator.simulate(recipe(c, name)).true for name, c in strategies.items()}

print(f"{'strategy':<20}{'peak VCD':>12}{'end viab.':>11}{'end vvd':>9}{'min DO':>8}{'harvest':>16}")
for name, r in results.items():
    harvested = (r.harvested_product[-1] + r.c_product[-1] * r.volume[-1]) * MOLAR_MASS
    print(
        f"{name:<20}{r.x_viable.max() / 1e6:>9.0f}e6{r.x_viable[-1] / r.x_total[-1]:>10.0%}"
        f"{r.feed_rate[-1] / r.volume[-1]:>9.2f}{100 * r.c_o2.min() / 0.2:>7.0f}%"
        f"{harvested / r.volume[-1]:>11.1f} g/L·V"
    )

In [ ]:
figure = make_subplots(
    rows=3, cols=2, shared_xaxes=True,
    subplot_titles=(
        "Viable cells (1e6/mL)", "Viability (%)",
        "Biomaterial (a.u.)", "Perfusion and bleed rate (vvd)",
        "Product in reactor (g/L)", "Product collected, harvest + reactor (g per L reactor)",
    ),
)
colors = ["#888888", "#1f77b4", "#2ca02c", "#d62728"]
for (name, r), color in zip(results.items(), colors):
    days = r.timepoints / 24.0
    show = dict(legendgroup=name, line=dict(color=color))
    figure.add_trace(go.Scatter(x=days, y=r.x_viable / 1e6, name=name, **show), 1, 1)
    figure.add_trace(go.Scatter(x=days, y=100 * r.x_viable / r.x_total, showlegend=False, **show), 1, 2)
    figure.add_trace(go.Scatter(x=days, y=r.biomaterial, showlegend=False, **show), 2, 1)
    figure.add_trace(go.Scatter(x=days, y=r.feed_rate / r.volume, showlegend=False, **show), 2, 2)
    figure.add_trace(
        go.Scatter(x=days, y=r.bleed_rate / r.volume, showlegend=False,
                   legendgroup=name, line=dict(color=color, dash="dot")), 2, 2)
    figure.add_trace(go.Scatter(x=days, y=r.c_product * MOLAR_MASS, showlegend=False, **show), 3, 1)
    collected = (r.harvested_product + r.c_product * r.volume) * MOLAR_MASS / r.volume
    figure.add_trace(go.Scatter(x=days, y=collected, showlegend=False, **show), 3, 2)
figure.update_xaxes(title_text="Time (days)", row=3)
figure.update_layout(height=850, width=1000, title="Batch vs. Perfusion (dotted: bleed)")
figure.show()

## 3. Your Own Control Loop
`simulator.start(recipe, seed)` returns a running reactor: read `session.measure()` (noisy measurements, as an operator or controller sees them) and apply flows with `session.step(action, hours)`. This is the interface for your own controllers: PI, MPC, reinforcement learning, ...

Example: a simple rule-based controller with typical noise: start perfusing at day 3, raise the perfusion rate when glucose drops, and bleed whenever the viable cell density exceeds 40e6 cells/mL.

In [ ]:
noisy = replace(recipe(ConstantRates(0.0, 0.0), "own controller"), noise_parameters=NoiseParameters.typical())
session = simulator.start(noisy, seed=7)
vvd = 0.0
while not session.done:
    m = session.measure()
    if m["time"] >= 72.0:
        vvd = min(4.0, max(1.0, vvd + (0.25 if m["c_glc"] < 10.0 else -0.05)))
    bleed = 0.3 if m["x_viable"] > 40e6 else 0.0
    action = ControlAction(feed_rate=vvd * session.volume, bleed_rate=min(bleed, vvd) * session.volume)
    session.step(action, hours=4.0)
own = session.result()

figure = make_subplots(rows=1, cols=3, subplot_titles=("Viable cells (1e6/mL)", "Glucose (mM)", "Flows (vvd)"))
days = own.true.timepoints / 24.0
figure.add_trace(go.Scatter(x=days, y=own.true.x_viable / 1e6, name="true"), 1, 1)
figure.add_trace(go.Scatter(x=days, y=own.measured.x_viable / 1e6, mode="markers", name="measured"), 1, 1)
figure.add_trace(go.Scatter(x=days, y=own.true.c_glc, showlegend=False), 1, 2)
figure.add_trace(go.Scatter(x=days, y=own.true.feed_rate / own.true.volume, name="perfusion"), 1, 3)
figure.add_trace(go.Scatter(x=days, y=own.true.bleed_rate / own.true.volume, name="bleed"), 1, 3)
figure.update_layout(height=380, width=1100)
figure.show()

## 4. Verification Against Richelle et al. (2022)
`presets.richelle_2022_celltype()` is the paper's cell population model (Table 1) inside the full simulator, with every other mechanism switched off. It reproduces the paper's equations (Eqs. 8-14), solved here directly (this is also a unit test).

In [ ]:
from scipy.integrate import solve_ivp

from dbiolab.mammalian_cellculture_simulation.presets import (
    RICHELLE_PH,
    RICHELLE_TEMPERATURE,
    richelle_2022_celltype,
    richelle_initial_conditions,
)

volume = 0.2  # L
richelle = MammalianRecipe(
    name="Richelle preset",
    celltype=richelle_2022_celltype(),
    temperature=RICHELLE_TEMPERATURE,
    ph_value=RICHELLE_PH,
    stirring_rate=200.0,
    initial_conditions=richelle_initial_conditions(),
    feeding_strategy=[],
    runtime=20 * 24.0,
    measurement_intervals=24.0,
    noise_parameters=NoiseParameters(),
    do_setpoint=None,
    perfusion=PerfusionSettings(
        medium=concentrations(c_glc=1.0, c_gln=1.0, c_o2=0.2),
        retention=RetentionDevice(cell_retention=1.0, product_passage=1.0, hcp_passage=1.0),
        controller=ConstantRates(feed_vvd=0.4 / volume, bleed_vvd=0.03 / volume),
        control_interval=24.0,
    ),
)
ours = MammalianCellcultureReactor(bench_perfusion_reactor(volume)).simulate(richelle).true

# The paper's equations (Eqs. 8-14) in its units: 1e6 cells/mL, days, L/day
mu_max, kd, kt, k_i, kl = 0.8384, 0.0209, 0.0290, 24.3905, 0.7743
d_feed, d_bleed = 0.4 / volume, 0.03 / volume


def richelle(_, y):
    xv, xd, xl, ob = y
    mu = mu_max / ((ob / k_i) ** 3 + 1.0)
    death = kd + kt * xl
    return [(mu - death - d_bleed) * xv, death * xv - (kl + d_bleed) * xd, kl * xd - d_feed * xl, xv - d_feed * ob]


paper = solve_ivp(richelle, (0, 20), [0.3, 0.023, 0.0, 0.0], t_eval=np.arange(21.0), rtol=1e-9, atol=1e-12)
figure = go.Figure()
for i, (name, ours_values) in enumerate(
    [("Xv", ours.x_viable / 1e6), ("Xd", ours.x_dead / 1e6), ("Xl", ours.x_lysed / 1e6)]
):
    figure.add_trace(go.Scatter(x=paper.t, y=paper.y[i], name=f"{name} paper equations", line=dict(width=5), opacity=0.4))
    figure.add_trace(go.Scatter(x=ours.timepoints / 24, y=ours_values, name=f"{name} cell culture simulator", mode="markers"))
figure.update_layout(title="Richelle et al. (2022): the simulator reproduces the paper's model",
                     xaxis_title="Time (days)", yaxis_title="1e6 cells/mL", height=420, width=900)
figure.show()
print("max. relative difference Xv:", np.max(np.abs(ours.x_viable / 1e6 - paper.y[0]) / paper.y[0]))